# 02 Preprocessing
Tokenization of the texts, the vocabulary, the coverage of the pretrained word embeddings, and the embedding matrix that both models use.

Load the train, validation and test splits (like on slide 9 of the course project slides).

In [8]:
from datasets import load_dataset

train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")
test = load_dataset("amarfurt/art-nlp-hs26", split="test")

print(len(train), len(valid), len(test))

16801 1000 1532


## Tokenization
NLTK's tokenization needs the `punkt_tab` resource, a pretrained model that `word_tokenize` uses to split text into sentences. It is downloaded first.


In [9]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mmici\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Compare splitting at whitespace with NLTK's `word_tokenize` on one example (the first short training hypothesis that contains both a comma and a contraction).


In [10]:
text = train[269]['hypothesis_1']
print(text)
print(text.split())
print(nltk.word_tokenize(text))

He didn't listen to them, and had to go to the hospital.
['He', "didn't", 'listen', 'to', 'them,', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital.']
['He', 'did', "n't", 'listen', 'to', 'them', ',', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital', '.']


.split() is a built-in Python string method. It cuts the text at every space and returns a list of pieces.
Punctuation stays attached to words, because there's no space between them and ,. `word_tokenize` separates punctuation (`them` and `,`) and contractions (`did` and `n't`) from the words. This way the words match the vocabulary of the word embeddings, and negations like `n't` stay their own tokens. `word_tokenize` is used for all texts.

In [12]:
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-300')


In [13]:
print(f'Vector size: {wv.vector_size}')
print(f'Vocabulary size: {len(wv.index_to_key)}')
for index, word in enumerate(wv.index_to_key[:10]):
    print(f"word #{index}: {word}")

Vector size: 300
Vocabulary size: 400000
word #0: the
word #1: ,
word #2: .
word #3: of
word #4: to
word #5: and
word #6: in
word #7: a
word #8: "
word #9: 's


Count the words in the training data. Tokenize all texts  (O1, O2, h1, h2) and count how many words there are in total (tokens) and how many distinct words (types), once as written and once lowercased. This gives you the list of words that need a vector.

In [17]:
columns = ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2']

tokens = []
for column in columns:
    for text in train[column]:
        tokens.extend(nltk.word_tokenize(text))

fdist = nltk.FreqDist(tokens)
fdist_lower = nltk.FreqDist(token.lower() for token in tokens)
print('Tokens:', len(tokens))
print('Types:', len(fdist))
print('Types (lowercased):', len(fdist_lower))
print(f'Type-token-ratio: {len(fdist) / len(tokens):.2%}')

Tokens: 655477
Types: 17789
Types (lowercased): 16210
Type-token-ratio: 2.71%


## Sources
- Dataset: course version of ART, `amarfurt/art-nlp-hs26` on Hugging Face (A. Marfurt), based on Bhagavatula et al. (2020), *Abductive Commonsense Reasoning*, ICLR 2020, https://arxiv.org/abs/1908.05739.
- Loading the splits: code from slide 9 of the course slides "Course Project HS26" (A. Marfurt).
- Tokenization: `nltk.download('punkt_tab')` and `nltk.word_tokenize` as in the course exercise "NLP Pipeline" (Week 1).
- Word embeddings: GloVe (Pennington et al., 2014), loaded with gensim's downloader (Řehůřek & Sojka, 2010) as in the course exercise "Word Embeddings" (Week 2).
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.